In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Новости по каждой компании

In [4]:
sber_news = pd.read_csv('Sber_news_last.csv')
sber_news['Company'] = ['sber' for i in range(len(sber_news))]
lukoil_news = pd.read_csv('lukoil_news_last_1.csv')
lukoil_news['Company'] = ['lukoil' for i in range(len(lukoil_news))]
nlmk_news = pd.read_csv('nlmk_news_last.csv')
nlmk_news['Company'] = ['nlmk' for i in range(len(nlmk_news))]
magnit_news = pd.read_csv('magnit_news_last.csv')
magnit_news['Company'] = ['magnit' for i in range(len(magnit_news))]
positiv_news = pd.read_csv('positive_tech_news_last.csv')
positiv_news['Company'] = ['positiv' for i in range(len(positiv_news))]

### Все новости

In [5]:
all_news = pd.concat([sber_news, lukoil_news, nlmk_news, magnit_news, positiv_news])

In [6]:
all_news['News_released'] = all_news['News_released'].apply(lambda x: x.split()[0])
all_news.rename(columns = {'News_released': 'Date'}, inplace = True) 
all_news['Date'] = all_news['Date'].apply(lambda x: pd.to_datetime(x)) # переведем все в datetime, чтобы работать как с дата объектом 
all_news = all_news.groupby(['Date', 'Company']).agg({'Header': ' '.join, 'Full_news': ' '.join}).reset_index()
all_news = all_news.sort_values(by = 'Date', ascending = False)

In [7]:
all_news

,Date,Company,Header,Full_news
2829,2024-12-23,magnit,Приобретение подконтрольной эмитенту организац...,Сообщение о существенном факте\nо приобретении...
2828,2024-12-20,positiv,Совершение эмитентом существенной сделки,Совершение эмитентом существенной сделки\n\n1....
2827,2024-12-20,lukoil,Проведение заседания совета директоров (наблюд...,СООБЩЕНИЕ О СУЩЕСТВЕННОМ ФАКТЕ\n О ПРОВЕДЕНИИ ...
2826,2024-12-19,positiv,Решения единственного акционера (участника),Решения единственного акционера (участника)\n\...
2825,2024-12-19,magnit,"Выплаченные доходы или иные выплаты, причитающ...",Сообщение\nо выплаченных доходах по ценным бум...
...,...,...,...,...
5,2018-01-10,sber,Выплаченные доходы по эмиссионным ценным бумаг...,СООБЩЕНИЕ о существенном факте \n«О выплаченны...
3,2018-01-08,sber,Дата начала размещения ценных бумаг Утверждени...,СООБЩЕНИЕ \nо дате начала размещения ценных бу...
2,2018-01-06,lukoil,"Сведения, направленные за пределы РФ для их ра...","Сообщение о существенном факте\nо сведениях, н..."
1,2018-01-03,lukoil,"Сведения, направленные за пределы РФ для их ра...","Сообщение о существенном факте\nо сведениях, н..."


### Токенизация 

In [173]:
import re

def tokenise(s):
    # Убираем пробелы внутри чисел (например, 10 000 000 → 10000000)
    s = re.sub(r'(?<=\d) (?=\d)', '', s)

    # Заменяем все вещественные числа и проценты на маркеры
    s_with_marks = re.sub(r'\d+\.\d+%', lambda m: f'PERCENT{m.group(0)}__', s)
    s_with_marks = re.sub(r'\d+\.\d+', lambda m: f'FLOAT{m.group(0)}__', s_with_marks)

    # Группируем "руб." с предыдущим числом или словом
    s_with_marks = re.sub(r'(\S+) (руб\.)', r'\1\2', s_with_marks)

    # Убираем запятые, если они не являются частью числа (например, "1,234" должно остаться)
    s_with_marks = re.sub(r'(?<!\d),(?!\d)', '', s_with_marks)

    # Разделяем строку по пробелам, знакам препинания (кроме точек в числах и процентах)
    parts = re.split(r'[^a-zA-Zа-яА-Я0-9.%]+', s_with_marks)

    # Убираем одинокие знаки препинания и точки в конце слов (если они не в числе/проценте)
    result = [
        re.sub(r'\.(?=$|[^0-9])', '', part.replace('FLOAT', '').replace('PERCENT', ''))
        for part in parts
        if part and not re.fullmatch(r'[.,%]+', part)
    ]

    # Восстанавливаем "не " перед словами
    cleaned_result = []
    i = 0
    while i < len(result):
        if result[i] == "не" and i + 1 < len(result):
            cleaned_result.append("не " + result[i + 1])
            i += 2  # Пропускаем следующее слово, так как уже объединили
        else:
            cleaned_result.append(result[i])
            i += 1

    # Фильтруем слова:
    final_result = [
        word for word in cleaned_result
        if "ru" not in word.lower()                 # Убираем слова с "ru"
        and "www" not in word.lower()               # Убираем слова с "www"
        and not re.search(r'\d+b', word.lower())    # Убираем облигации (например, "10b")
        and not re.search(r'\d+[a-zA-Z]+', word)    # Убираем смешанные слова (например, "4b0215401481b001p")
    ]

    return final_result

In [177]:
all_news['Full_news'] = all_news['Full_news'].apply(lambda x: tokenise(str(x).lower()))
all_news.head()

,Date,Company,Header,Full_news
2829,2024-12-23,magnit,Приобретение подконтрольной эмитенту организац...,"[сообщение, о, существенном, факте, о, приобре..."
2828,2024-12-20,positiv,Совершение эмитентом существенной сделки,"[совершение, эмитентом, существенной, сделки, ..."
2827,2024-12-20,lukoil,Проведение заседания совета директоров (наблюд...,"[сообщение, о, существенном, факте, о, проведе..."
2826,2024-12-19,positiv,Решения единственного акционера (участника),"[решения, единственного, акционера, участника,..."
2825,2024-12-19,magnit,"Выплаченные доходы или иные выплаты, причитающ...","[сообщение, о, выплаченных, доходах, по, ценны..."


### Лемматизация

In [178]:
import nltk
from nltk import WordNetLemmatizer
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

In [179]:
def lemmatizing(tokenized_text):
    text = [morph.parse(word)[0].normal_form for word in tokenized_text]
    return text

all_news['Full_news'] = all_news['Full_news'].apply(lambda x: lemmatizing(x))
all_news.head()

,Date,Company,Header,Full_news
2829,2024-12-23,magnit,Приобретение подконтрольной эмитенту организац...,"[сообщение, о, существенный, факт, о, приобрет..."
2828,2024-12-20,positiv,Совершение эмитентом существенной сделки,"[совершение, эмитент, существенный, сделка, 1,..."
2827,2024-12-20,lukoil,Проведение заседания совета директоров (наблюд...,"[сообщение, о, существенный, факт, о, проведен..."
2826,2024-12-19,positiv,Решения единственного акционера (участника),"[решение, единственный, акционер, участник, 1,..."
2825,2024-12-19,magnit,"Выплаченные доходы или иные выплаты, причитающ...","[сообщение, о, выплатить, доход, по, ценный, б..."


### Стоп слова

In [181]:
import nltk

nltk.download('stopwords')
stopword = nltk.corpus.stopwords.words('russian')
stopword_new = ['сообщение', 'иных', 'существенное', 'существенный', 'факт','событие', 'влияние', 'существенном', 'действие', 'факте', 'протокол',
               'орган', ',', 'выпуск', 'ru', 'номер', 'bsk', 'prt', 'далее', 'серия', 'b', 't', '4b02', 'prt', 'www', '401481в001p02e',
               '5y', 'категория', 'isin', 'http', '001р', 'mem', 'мнение', 'кворум', 'имелся','запятая',
               'пао', 'fix', 'федерация', 'должный', 'фирменный', 'страница', 'признак', 'указанный', 'иос', 'адрес', 'форма', 'инна', 'иной', 'г']
stopword.extend(stopword_new)
stopword.remove('не')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [182]:
def remove_stopwords(tokenised_list):
    text = [word for word in tokenised_list if word not in stopword]
    return text

all_news['Full_news'] = all_news['Full_news'].apply(lambda x: remove_stopwords(x))
all_news.head()

,Date,Company,Header,Full_news
2829,2024-12-23,magnit,Приобретение подконтрольной эмитенту организац...,"[приобретение, голосовать, акция, доля, эмитен..."
2828,2024-12-20,positiv,Совершение эмитентом существенной сделки,"[совершение, эмитент, сделка, 1, общий, сведен..."
2827,2024-12-20,lukoil,Проведение заседания совета директоров (наблюд...,"[проведение, заседание, совет, директор, эмите..."
2826,2024-12-19,positiv,Решения единственного акционера (участника),"[решение, единственный, акционер, участник, 1,..."
2825,2024-12-19,magnit,"Выплаченные доходы или иные выплаты, причитающ...","[выплатить, доход, ценный, бумага, эмитент, та..."


### Сделали разделение с исключением 20% самых часто встречающихся слов

In [183]:
all_news['Full_news'] = all_news['Full_news'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

In [185]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Создаем TF-IDF векторизатор (удаляем редкие и частые слова)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.8)
tfidf_matrix = vectorizer.fit_transform(all_news['Full_news'])

# Получаем список оставшихся слов/фраз
remaining_words = set(vectorizer.get_feature_names_out())

# Фильтруем тексты: удаляем слова, не попавшие в TF-IDF
def filter_text(text):
    words = text.split()  # Разбиваем текст на слова
    filtered_words = [word for word in words if word in remaining_words]  # Оставляем только нужные слова
    return ' '.join(filtered_words)  # Собираем обратно

all_news['filtered_text'] = all_news['Full_news'].apply(filter_text)

### Все котировки акций

In [187]:
sber_stocks = pd.read_csv('Прошлые данные - SBER.csv')
sber_stocks['Company'] = ['sber' for i in range(len(sber_stocks))]
lukoil_stocks = pd.read_csv("Прошлые данные - LKOH.csv")
lukoil_stocks['Company'] = ['lukoil' for i in range(len(lukoil_stocks))]
nlmk_stocks = pd.read_csv("Прошлые данные - NLMK.csv")
nlmk_stocks['Company'] = ['nlmk' for i in range(len(nlmk_stocks))]
magnit_stocks = pd.read_csv("Прошлые данные - MGNT.csv")
magnit_stocks['Company'] = ['magnit' for i in range(len(magnit_stocks))]
positiv_stocks = pd.read_csv("Прошлые данные - POSI.csv")
positiv_stocks['Company'] = ['positiv' for i in range(len(positiv_stocks))]

In [188]:
all_stocks = pd.concat([sber_stocks, lukoil_stocks, nlmk_stocks, magnit_stocks, positiv_stocks])

In [189]:
all_stocks['Дата'] = all_stocks['Дата'].apply(lambda x: x.split()[0])
all_stocks.rename(columns = {'Дата': 'Date'}, inplace = True)
all_stocks['Date'] = all_stocks['Date'].apply(lambda x: pd.to_datetime(x)) # переведем все в datetime, чтобы работать как с дата объектом 
all_stocks = all_stocks.sort_values(by = 'Date', ascending = False)

In [190]:
all_stocks

,Date,Цена,Откр.,Макс.,Мин.,Объём,Изм. %,Company
0,2024-12-24,"264,34","264,94","266,62","261,03","83,81M","-0,21%",sber
0,2024-12-24,"4.945,5","4.984,5","4.984,5","4.872,5","504,40K","-0,77%",magnit
0,2024-12-24,"133,66","135,00","136,10","131,46","9,97M","-0,33%",nlmk
0,2024-12-24,"1.901,00","2.019,00","2.011,40","1.891,00","1,08M","-5,73%",positiv
0,2024-12-24,"6.833,0","6.919,0","6.959,5","6.825,0","724,13K","-1,24%",lukoil
...,...,...,...,...,...,...,...,...
1704,2018-01-03,"4.985,0","4.770,0","4.995,0","4.701,0","900,16K","4,25%",magnit
1739,2018-01-02,"263,20","265,00","270,30","260,60","73,04M","-0,49%",sber
1723,2018-01-02,"5.219,0","5.338,0","5.368,0","5.174,0","661,83K","-2,08%",magnit
1723,2018-01-02,"3.760,5","3.733,0","3.762,5","3.722,5","506,78K","0,87%",lukoil


In [191]:
for col in ['Цена', 'Откр.', 'Макс.', 'Мин.', 'Изм. %']:
    all_stocks[col] = all_stocks[col].apply(lambda x: x.replace('.', '') if '.' in x else x)
    all_stocks[col] = all_stocks[col].apply(lambda x: float(x.replace(',', '.').strip('%')))

In [192]:
all_stocks

,Date,Цена,Откр.,Макс.,Мин.,Объём,Изм. %,Company
0,2024-12-24,264.34,264.94,266.62,261.03,"83,81M",-0.21,sber
0,2024-12-24,4945.50,4984.50,4984.50,4872.50,"504,40K",-0.77,magnit
0,2024-12-24,133.66,135.00,136.10,131.46,"9,97M",-0.33,nlmk
0,2024-12-24,1901.00,2019.00,2011.40,1891.00,"1,08M",-5.73,positiv
0,2024-12-24,6833.00,6919.00,6959.50,6825.00,"724,13K",-1.24,lukoil
...,...,...,...,...,...,...,...,...
1704,2018-01-03,4985.00,4770.00,4995.00,4701.00,"900,16K",4.25,magnit
1739,2018-01-02,263.20,265.00,270.30,260.60,"73,04M",-0.49,sber
1723,2018-01-02,5219.00,5338.00,5368.00,5174.00,"661,83K",-2.08,magnit
1723,2018-01-02,3760.50,3733.00,3762.50,3722.50,"506,78K",0.87,lukoil


### Объединяем датафреймы по дате и названию компании
#### how = 'outer' -> если нет новости получаем NaN значение, если нет информации по торгам, также NaN значение

In [202]:
all_merged = pd.merge(all_news, all_stocks, on = ['Date', 'Company'], how='outer')
all_merged['Date'] = all_merged['Date'].apply(lambda x: pd.to_datetime(x))

In [203]:
all_merged = all_merged.sort_values(by = 'Date', ascending = False)
all_merged.index = range(len(all_merged))

In [204]:
all_merged['Day'] = all_merged['Date'].dt.day_name()
all_merged = all_merged[['Date', 'Day', 'Company', 'Header', 'filtered_text', 'Цена', 'Откр.', 'Макс.', 'Мин.', 'Объём', 'Изм. %']]

In [205]:
all_merged

,Date,Day,Company,Header,filtered_text,Цена,Откр.,Макс.,Мин.,Объём,Изм. %
0,2024-12-24,Tuesday,magnit,NaN,NaN,4945.50,4984.50,4984.50,4872.50,"504,40K",-0.77
1,2024-12-24,Tuesday,sber,NaN,NaN,264.34,264.94,266.62,261.03,"83,81M",-0.21
2,2024-12-24,Tuesday,nlmk,NaN,NaN,133.66,135.00,136.10,131.46,"9,97M",-0.33
3,2024-12-24,Tuesday,positiv,NaN,NaN,1901.00,2019.00,2011.40,1891.00,"1,08M",-5.73
4,2024-12-24,Tuesday,lukoil,NaN,NaN,6833.00,6919.00,6959.50,6825.00,"724,13K",-1.24
...,...,...,...,...,...,...,...,...,...,...,...
6835,2018-01-03,Wednesday,magnit,NaN,NaN,4985.00,4770.00,4995.00,4701.00,"900,16K",4.25
6836,2018-01-02,Tuesday,lukoil,NaN,NaN,3760.50,3733.00,3762.50,3722.50,"506,78K",0.87
6837,2018-01-02,Tuesday,magnit,Проведение заседания совета директоров (наблюд...,проведение заседание совет директор наблюдател...,5219.00,5338.00,5368.00,5174.00,"661,83K",-2.08
6838,2018-01-02,Tuesday,sber,NaN,NaN,263.20,265.00,270.30,260.60,"73,04M",-0.49


In [206]:
all_merged.to_csv('all_merged.csv')

### Добавим изменение логарифмической доходности с лагом в 1,2,3 дня
#### Также мы поочередно в новый датафрейм добавили упорядоченные по времени данные о каждой из компаний, чтобы не обучать 5 моделей для каждой компании

In [288]:
final_df = pd.DataFrame()
for company in all_merged['Company'].unique():    
    company_df = all_merged[all_merged['Company'] == f'{company}'].copy()
    company_df['Изменение с лагом в 3 дня %'] = (np.log(company_df['Цена'].shift(3) / company_df['Цена'])) * 100
    company_df['Изменение с лагом в 2 дня %'] = (np.log(company_df['Цена'].shift(2) / company_df['Цена'])) * 100
    company_df['Изменение с лагом в 1 день %'] = (np.log(company_df['Цена'].shift(1) / company_df['Цена'])) * 100
    final_df = pd.concat([final_df, company_df], ignore_index=True)
final_df

,Date,Day,Company,Header,filtered_text,Цена,Откр.,Макс.,Мин.,Объём,Изм. %,Изменение с лагом в 3 дня %,Изменение с лагом в 2 дня %,Изменение с лагом в 1 день %
0,2024-12-24,Tuesday,magnit,NaN,NaN,4945.5,4984.5,4984.5,4872.5,"504,40K",-0.77,NaN,NaN,NaN
1,2024-12-23,Monday,magnit,Приобретение подконтрольной эмитенту организац...,приобретение голосовать акция доля депозитарны...,4984.0,4991.5,5081.0,4926.0,"753,38K",1.27,NaN,NaN,-0.775471
2,2024-12-20,Friday,magnit,NaN,NaN,4921.5,4398.0,4923.0,4376.0,"1,08M",11.46,NaN,0.486471,1.261942
3,2024-12-19,Thursday,magnit,"Выплаченные доходы или иные выплаты, причитающ...",выплатить доход также выплата причитаться влад...,4415.5,4452.0,4521.5,4407.0,"451,74K",-0.38,11.335699,12.111170,10.849228
4,2024-12-18,Wednesday,magnit,NaN,NaN,4432.5,4350.0,4466.5,4301.0,"398,91K",2.13,11.726902,10.464960,-0.384268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6835,2018-01-10,Wednesday,lukoil,NaN,NaN,5016.0,5045.0,5115.0,4980.0,"944,92K",-0.12,-26.144762,-25.770012,-1.587494
6836,2018-01-08,Monday,lukoil,NaN,NaN,4494.0,4460.0,4509.0,4454.5,"513,11K",0.95,-14.781049,9.401469,10.988963
6837,2018-01-06,Saturday,lukoil,"Сведения, направленные за пределы РФ для их ра...",направлять предоставлять соответствовать иност...,4205.5,4210.0,4258.0,4180.5,"750,95K",-0.08,16.036467,17.623961,6.634998
6838,2018-01-03,Wednesday,lukoil,"Сведения, направленные за пределы РФ для их ра...",направлять предоставлять соответствовать иност...,3754.0,3767.5,3790.0,3741.0,"581,00K",-0.60,28.981087,17.992124,11.357125


In [289]:
final_df.to_csv('final_df.csv')